# 6.2 对抗性数据与不变性

### 对抗性攻击
对抗性攻击（Adversarial Attacks）：是指通过对输入数据进行微小的、人类难以察觉的修改，从而诱导机器学习模型做出错误判断的行为。

**数学原理**

攻击原理：训练数据和“自然”的测试数据通常只存在于高维空间中一个非常小的子集（Support）上。在未定义区域上，模型的函数行为是未知的。

训练过程：调整参数 $w$ 来减小损失

攻击操作：保持模型参数不变，调整输入 $x$ 来最大化损失
$$\text{maximize } l(f(x+\delta), y), \ \text{subject to } ||\delta||\le \epsilon$$
- $\delta$（扰动）：这是攻击者添加的修改
- $x+\delta$：这是修改后的“对抗样本” 
- $||\delta|| \le \epsilon$：这是一个约束条件，确保修改非常微小，以至于人类视觉或听觉无法察觉 。


### 攻击手段

1. 梯度类攻击 (Gradient-based Attacks)

    利用模型的“梯度”信息，寻找能让 Loss（损失函数）增长最快的方向

    - FGSM (快速梯度符号法)：这是一种“一击制胜”的方法。攻击者计算 Loss 对输入 $x$ 的梯度，然后沿着梯度的方向挪动一小步。因为它只计算一次梯度，所以速度极快。

    - PGD (投影梯度下降)：这是 FGSM 的增强版。它不是一次迈大步，而是分很多次迈小步，且每一步之后都会检查是否超出了约定的扰动范围 $\epsilon$ 。如果超出了，就把它“投影”回合法的范围内。

2. 优化类攻击 (Optimization-based Attacks)

    更隐蔽的攻击手段：为了让模型彻底改判成我想要的类别，我需要添加的最微小、最隐蔽的扰动 $\delta$ 是多少

    - Carlini & Wagner (C&W) 攻击：
        - 它会同时寻找两个目标的平衡点：一是让扰动 $\delta$ 尽可能小（肉眼看不见），二是让分类结果尽可能偏向错误的类别。
        - $Loss_{total} = ||\delta||_p + c \cdot f(x + \delta)$
            1. 扰动项 $||\delta||_p$（隐蔽性）：越小越不易被察觉
            2. 攻击项 $f(x + \delta)$（成功率）：这部分是一个特别设计的函数，当模型还没有被骗过时，它的值很大；一旦模型被骗过了，它的值就会变为 0 或很小。
            3. 常数 $c$：一个权重，用来平衡“改动小”和“骗过模型”这两个矛盾的目标
        
    
    - 物理世界攻击：攻击者在设计扰动时，会考虑光照、角度等物理因素，确保即使在摄像头捕捉到的真实画面中，攻击依然有效。



### 数据增强与不变性 (Data Augmentation & Invariances)

不变性：即使对数据进行某些转换，其标签（Label）也不应该发生变化

常见的不变性变换：
- 缩放（Scaling）与旋转（Rotation） 
- 轴变形（Axis deformation）与厚度变形（Thickness deformation）

| 数据类型          | 常见增强手段     |
|-------------------|----------------------------------------------------------------------|
| 图像 (Images) 📸  | 水平/垂直翻转、随机裁剪 (Crop)、调整颜色（亮度、饱和度、色调）|
| 语音 (Speech) 🎙️ | 添加背景噪音、改变音调或语速|
| 文本 (Text) 📄    | 随机移除词语、插入词语、同义词替换、随机子串   |

在线生成（Generate on the fly）：
- 实际训练中，通常在模型训练的过程中实时生成增强数据，而非预先增强后保存下来。
- 好处：模型在每个训练轮次（Epoch）看到的样本都是略有不同的，从而极大地提高了模型的泛化能力和鲁棒性 。


**切线距离（Tangent Distance）**：一种衡量变换后样本间距离的方法

- 将一张图像 $I$ 看作高维空间中的一个点
- 当我们对它进行某种变换（例如旋转）时，随着变换参数 $\alpha$（如角度）的变化，这个点会在空间中移动，画出一条曲线。这条曲线在数学上被称为流形（Manifold） 。
- 切向量（Tangent Vector）就是这条曲线在原始图像点处的“切线方向”。它代表了图像随变换参数变化的瞬时变化率。

    假设变换后的图像函数为 $T(I, \alpha)$，其中 $\alpha$ 是变换强度，那么对应的切向量 $v$ 就是对参数 $\alpha$ 的偏导数：
    $$v = \frac{\partial T(I, \alpha)}{\partial \alpha} \bigg|_{\alpha=0}$$

    设图片 $I$ 对某种变换的一组切向量为 $\{v_1, v_2, ..., v_k\}$，则微小变换后的图片 $T(I, \alpha) = I + \sum_{i=1}^k \alpha_i v_i$

切线距离通常通过以下方式融入模型：

- 改进分类算法：在 K-最近邻（K-NN） 算法中，用切线距离代替欧氏距离，可以显著提升模型对旋转、缩放等变换的鲁棒性。

- 核函数（Kernel Machines）：在支持向量机（SVM）中，可以将切线距离嵌入核函数中，构建所谓的“切线距离核”。

- 对抗防御：在面对对抗性攻击时，切线距离可以帮助识别那些虽然像素值变了、但本质上只是沿着“切线方向”移动的正常样本，从而区分出真正的恶意扰动

### 对抗性训练与鲁棒损失 (Adversarial Training & Robust Loss)

**Convex loss**：

$$L(x,y,f) = \sup_{\delta \in \Lambda} \eta(\delta) l(f(x+\delta), y)$$

- $\sup_{\delta \in \Lambda}$（寻找最坏情况）：在训练的每一步，模型要在允许的变换范围 $\Lambda$ 内，找到那个能让损失 $l$ 达到最大（即最能误导模型）的扰动 $\delta$ 。
- $\eta(\delta)$（变换惩罚）：这是一个惩罚项。如果变换 $\delta$ 太过极端（比如把图片改得面目全非），$\eta(\delta)$ 的值会减小，从而降低这个极端例子对模型的影响 。
- 对抗网络（Adversarial Networks）：在实际操作中，通常会有一个“对抗样本生成器”专门负责寻找这个最坏的情况 。

**实质: Min-Max 博弈**

- 内部最大化（Inner Maximize）：对手在寻找能让模型出错的“最强攻击” $\delta$。
- 外部最小化（Outer Minimize）：模型更新自己的参数，目的是即使面对这些“最强攻击”，损失依然最小 。